# RaCo keypoint detection demo
Port of [cvg/RaCo `demo.ipynb`](https://github.com/cvg/RaCo/blob/main/demo.ipynb) at commit `ee06fa68`. The weights (`raco.pth`, official v1.0.0 release) and the sample images are loaded from `@RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS`, not downloaded at runtime.

In [ ]:
# Pin numpy so OpenCV doesn't upgrade it past the preinstalled scipy/numba
%pip install --quiet kornia opencv-python-headless "numpy<2.4"
%pip install --quiet --no-deps git+https://github.com/cvg/RaCo.git@ee06fa68dbb20303f334fbc257dad71d4149603b

In [ ]:
from pathlib import Path
from snowflake.snowpark.context import get_active_session

session = get_active_session()
STAGE = '@RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS'
ASSET_DIR = Path('/tmp/raco_demo')
(ASSET_DIR / 'assets').mkdir(parents=True, exist_ok=True)
session.file.get(f'{STAGE}/raco.pth', str(ASSET_DIR))
session.file.get(f'{STAGE}/assets/i_castle.png', str(ASSET_DIR / 'assets'))
WEIGHTS = str(ASSET_DIR / 'raco.pth')
print(sorted(p.name for p in ASSET_DIR.rglob('*') if p.is_file()))

In [ ]:
import torch
from raco import RaCo
from raco.utils import load_image

device = "cuda" if torch.cuda.is_available() else "cpu"
image = load_image(ASSET_DIR / "assets/i_castle.png").to(device)

# Initialize a RaCo extractor with the default configuration (weights from stage)
extractor = RaCo(max_num_keypoints=1024, weights=WEIGHTS).to(device)

In [ ]:
output = extractor.extract(image)
print("Output keys:", [k for k in output])

In [ ]:
import matplotlib.pyplot as plt

from raco import utils, viz2d

output = utils.batch_to_device(output, "cpu")
image = image.cpu()

ax = viz2d.plot_images([image])
viz2d.plot_keypoints([output["keypoints"][0]], axes=ax)

plt.suptitle("RaCo Keypoints", fontsize=16, y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
ax = viz2d.plot_images([image])

# Subsample keypoints for cleaner visualization
subsample = 10
idxs = torch.randperm(
    len(output["keypoints"][0]), generator=torch.Generator().manual_seed(42)
)[::subsample]

keypoints = output["keypoints"][0]
subsampled_keypoints = output["keypoints"][0][idxs]
subsampled_covariances = output["covariances"][0][idxs]

viz2d.plot_covariance_ellipses(
    [subsampled_keypoints],
    [subsampled_covariances],
    axes=ax,
    sigma=20,
)

viz2d.plot_keypoints(
    [keypoints],
    axes=ax,
)

plt.suptitle("RaCo Keypoint Detection with Uncertainties", fontsize=16, y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
from raco.utils import rank_from_scores

ax = viz2d.plot_images([image] * 2)

ranker_integer_ranks = rank_from_scores(output["ranker_scores"][0])
ranker_colors = viz2d.cm_GnRd(
    (ranker_integer_ranks - 1) / (len(ranker_integer_ranks) - 1)
)  # Normalize to [0, 1]

keypoint_score_integer_ranks = rank_from_scores(output["keypoint_scores"][0])
colors_keypoint_scores = viz2d.cm_GnRd(
    (keypoint_score_integer_ranks - 1) / (len(keypoint_score_integer_ranks) - 1)
)  # Normalize to [0, 1]

viz2d.plot_keypoints(
    [output["keypoints"][0]] * 2,
    axes=ax,
    colors=[ranker_colors, colors_keypoint_scores],
    ps=10,
)

plt.suptitle(
    "RaCo Keypoints Colored by Ranking, "
    f"Rank 1: Green, Rank {len(ranker_integer_ranks)}: Red",
    fontsize=16,
    y=0.95,
)
ax[0].set_title("RaCo Ranker Ranks")
ax[1].set_title("RaCo Keypoint Score Ranks")
plt.tight_layout()
plt.show()

Upstream's last cell loads the model with `torch.hub.load("cvg/RaCo", "raco", pretrained=True)`, which downloads the code and weights from GitHub at runtime. It's left out here so the notebook only uses the staged checkpoint.